# Full Training Notebook
This notebook runs the full fine-tuning pipeline for Qwen2-VL using LoRA and the same collator used in `train.ipynb`. Adjust `training_args` below for longer runs.

### Dataset loading
This cell loads the dataset JSON files (`train.json`, `valid.json`, `test.json`) from the `weapon VLM dataset` directory into a `datasets.DatasetDict` named `dataset`. Run this first to make data available to the rest of the notebook.

In [2]:
from pathlib import Path
from datasets import load_dataset
DATA_DIR = Path("2-class-weapon-dataset-3")
dataset = load_dataset(
    "json",
    data_files={
        "train": str(Path("Dataset Annotation-20260807T120404Z-1-001/Dataset Annotation/train.json")),
        "validation": str(Path("Dataset Annotation-20260807T120404Z-1-001/Dataset Annotation/valid.json")),
        "test": str(Path("Dataset Annotation-20260807T120404Z-1-001/Dataset Annotation/test.json")),
    },
)

c:\Users\User\.conda\envs\qwen-vl_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Model & LoRA setup
This cell loads the Qwen2-VL base model and `AutoProcessor`, configures 4-bit quantization, and wraps the model with LoRA (`get_peft_model`) using `lora_config`. Run after dataset is loaded to prepare `model` and `processor` for training.

In [3]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = "Qwen/Qwen3-VL-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(model_id)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

W0812 08:42:00.958000 11492 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\User\.conda\envs\qwen-vl_env\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--Qwen--Qwen3-VL-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-deve

### Prompt & image helpers
Defines `resolve_image_path`, `build_prompt_and_images`, and prompt tokens. These helpers assemble the text prompt and load associated images for each example. They are used by the collator and generation utilities.

In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Q1flzyg1J8Z2OMweLXf3")
project = rf.workspace("weapon-mpr3p").project("2-class-weapon-dataset")
version = project.version(3)
roboflow_dataset = version.download("yolo26")

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
loading Roboflow workspace...
loading Roboflow project...


In [5]:
from PIL import Image

SYSTEM_PROMPT = "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
ASSISTANT_PREFIX = "<|im_start|>assistant\n"
ASSISTANT_SUFFIX = "<|im_end|>"

def resolve_image_path(image_path):
    candidate = Path(image_path)
    if candidate.is_file():
        return candidate

    dataset_candidate = DATA_DIR / candidate
    if dataset_candidate.is_file():
        return dataset_candidate

    basename_candidate = DATA_DIR / candidate.name
    if basename_candidate.is_file():
        return basename_candidate

    for split_name in ("train", "valid", "test"):
        split_candidate = DATA_DIR / split_name / "images" / candidate.name
        if split_candidate.is_file():
            return split_candidate

    raise FileNotFoundError(f"Could not resolve image path: {image_path}")

def build_prompt_and_images(example, add_generation_prompt):
    messages = example["messages"]
    user_message = next(message for message in messages if message["role"] == "user")
    assistant_message = next((message for message in messages if message["role"] == "assistant"), None)

    image_paths = []
    text_parts = []
    for content_item in user_message["content"]:
        if content_item["type"] == "image":
            image_paths.append(resolve_image_path(content_item["image"]))
        elif content_item["type"] == "text":
            text_parts.append(content_item["text"])

    image_tokens = "".join("<|vision_start|><|image_pad|><|vision_end|>" for _ in image_paths)
    user_text = "".join(text_parts)
    prompt = SYSTEM_PROMPT + f"<|im_start|>user\n{image_tokens}{user_text}<|im_end|>\n"

    images = [Image.open(image_path).convert("RGB") for image_path in image_paths]

    if add_generation_prompt:
        return prompt + ASSISTANT_PREFIX, images

    assistant_text = "".join(
        content_item["text"]
        for content_item in (assistant_message["content"] if assistant_message is not None else [])
        if content_item.get("type") == "text"
    )
    return prompt + ASSISTANT_PREFIX + assistant_text + ASSISTANT_SUFFIX, images

### Collator (tokenization + label masking)
`collate_fn` tokenizes the full (prompt+assistant) sequence and the prompt-only sequence, then masks prompt tokens in `labels` with `-100` so the loss is computed only on the assistant-generated tokens. This cell must run before training.

In [6]:
import torch
def collate_fn(batch):
    prompt_texts = []
    full_texts = []
    images = []

    for example in batch:
        prompt_text, example_images = build_prompt_and_images(example, add_generation_prompt=True)
        full_text, _ = build_prompt_and_images(example, add_generation_prompt=False)

        prompt_texts.append(prompt_text)
        full_texts.append(full_text)
        images.append(example_images)

    tokenized_full = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True
    )

    tokenized_prompt = processor(
        text=prompt_texts,
        images=images,
        return_tensors="pt",
        padding=True
    )

    labels = tokenized_full["input_ids"].clone()

    prompt_mask_lengths = tokenized_prompt["attention_mask"].sum(dim=1)
    for i, plen in enumerate(prompt_mask_lengths.tolist()):
        if plen > 0:
            labels[i, : int(plen)] = -100

    tokenized_full["labels"] = labels

    return tokenized_full

### Training config & Trainer
This cell builds `TrainingArguments` tuned to the smoke results (safe learning rate, warmup, weight decay) and creates a `Trainer` that handles the training loop. Adjust `num_train_epochs`, `per_device_train_batch_size`, and `eval_steps` here for longer runs.

In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qwen3_vl_model_full3_ra1632",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    dataloader_num_workers=0,
    num_train_epochs=10,
    learning_rate=5e-5,
    warmup_steps=200,
    weight_decay=0.01,
    bf16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    remove_unused_columns=False,
    label_names=["labels"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_grad_norm=1.0,
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collate_fn,
)

trainer.train()

trainer.save_model("./qwen3_vl_model_full3_ra1632")
processor.save_pretrained("./qwen3_vl_model_full3_ra1632")


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
200,0.058031,0.020853
400,0.037513,0.022567
600,0.018511,0.021857
800,0.024129,0.026589
1000,0.017550,0.031370
1200,0.016356,0.034430
1400,0.008647,0.033466
1600,0.014469,0.026921
1800,0.006528,0.030781
2000,0.003431,0.041401


['./qwen3_vl_model_full3_ra1632\\processor_config.json']

### Model & processor save
After training completes, this cell saves the fine-tuned model and processor to `./qwen_vl_model_full` for later loading or inference.

In [8]:
# Manual validation pass (no Trainer callbacks)
from torch.utils.data import DataLoader
validation_loader = DataLoader(dataset["validation"], batch_size=2, shuffle=False, collate_fn=collate_fn)
model.eval()
val_losses = []
with torch.no_grad():
    for batch in validation_loader:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        out = model(**batch)
        val_losses.append(out.loss.item())

print('Validation batches:', len(val_losses))
print('Mean validation loss:', sum(val_losses)/len(val_losses) if val_losses else float('nan'))

Validation batches: 542
Mean validation loss: 0.0998839077012329
